<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">
  <div style="display: flex; align-items: center; gap: 16px; padding: 0 32px;">
    <div>
      <h1 style="margin:0; font-size:1.6em;">B1 · Clasificador Hídrico y Natural</h1>
      <p style="margin:4px 0 0; color: gray;">Clasificador multietiqueta de publicaciones hídricas y naturales en boletines oficiales españoles</p>
    </div>
  </div>
</div>


# B1 · Clasificador Hídrico y Natural

Este notebook implementa el clasificador del **bloque B1** (hídrico y natural) para el proyecto de monitorización de boletines oficiales españoles.

**Dominio**: publicaciones relacionadas con concesiones de aguas, vías pecuarias, montes de utilidad pública, espacios naturales protegidos, gestión de residuos, vertidos y planes hidrológicos.

**Arquitectura**: B1 comparte con B0 el pre-procesamiento N0/N1 (`agent.py`) pero tiene su propio schema (`schema_B1.py`) y prompts (`prompts_B1.py`).

| Capa | Qué etiqueta | Cómo |
|------|-------------|------|
| N0 | Ámbito geográfico | lookup por boletín, sin LLM |
| N1 | Tipo de acto | reglas de primer token, sin LLM |
| N2 | Categoría hídrica/natural | LLM (11 etiquetas) |
| N3 | Subcategoría uso/cuenca | LLM (15 etiquetas) |


---

##  0. Setup

Cargamos variables de entorno e importamos librerías. El modelo vive en LM Studio.

In [ ]:
# Librerías estándar
import os
import html
import re
import json
import asyncio
from pathlib import Path

# Datos
import pandas as pd
import numpy as np

# Métricas multilabel
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)

# Carga de variables de entorno desde .env (busca hacia arriba desde el directorio actual)
from dotenv import find_dotenv, load_dotenv

# Pydantic AI - framework que fuerza al LLM a devolver JSON validado por Pydantic
from pydantic_ai import Agent

SEED = 29092025

load_dotenv(find_dotenv())


In [ ]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

# Cambiar LM_STUDIO_MODEL según el modelo cargado en LM Studio
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)


In [ ]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"OK: {LM_STUDIO_MODEL} listo  |  otros: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - arrancar el servidor antes de continuar")


---

##  1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo,
de qué tipo son y qué invariantes deben cumplirse.

B1 reutiliza `ActType` de B0 (mismo enum de formas jurídicas) y define sus propios enums
`CategoryType` (N2) y `SubcategoryType` (N3).


In [ ]:
# Importar los tipos del schema B1: enums de etiquetas y modelo de output
from clasificador.schema_B1 import ActType, CategoryType, SubcategoryType, ClassifierOutput


In [ ]:
# Verificación de invariantes del schema B1
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    categories=[CategoryType.AGU_RIE],
    subcategories=[SubcategoryType.USO_AGRICOLA, SubcategoryType.CUENCA_DUERO],
    confidence=0.95,
    reasoning="'concesión de aguas para riego' → AGU_RIE. 'Confederación Hidrográfica del Duero' → cuenca_duero.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False, act_type=ActType.RESOLUCION,
        categories=[CategoryType.AGU_GEN], subcategories=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError -> {e.errors()[0]['msg']}")


---

##  2. Pre-procesamiento

El pre-procesamiento N0/N1 es **compartido entre todos los bloques** y vive en `agent.py`.
No se reimplementa aquí: solo se importa.

- **N0**: ámbito geográfico por boletín (`get_ambito`)
- **N1**: tipo de acto por reglas de primer token (`inferir_act_type`)


In [ ]:
# Funciones N0/N1 compartidas con B0 - viven en agent.py
from clasificador.agent import get_ambito, inferir_act_type

# Cargar el corpus completo
PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
print(f"Corpus total: {len(df):,} registros | Columnas: {list(df.columns)}")


In [ ]:
# Aplicar clasificador N1 (reglas de primer token) al corpus completo
df["act_type_n1"] = df.apply(
    lambda r: inferir_act_type(str(r["description"]), str(r["bulletin"])).value, axis=1
)

# Filtrar al dominio B1 usando las keywords de muestreo para estimar el tamaño del universo
keywords_dominio = [
    "confederación hidrográfica", "comisaría de aguas", "aprovechamiento de aguas",
    "regadío", "comunidad de regantes", "riego", "sondeo para captación",
    "captación de aguas subterráneas", "abastecimiento de agua", "abastecimiento municipal",
    "aprovechamiento hidroeléctrico", "central hidroeléctrica",
    "vía pecuaria", "cañada real", "cordel", "vereda",
    "monte de utilidad pública", "dominio público forestal",
    "parque natural", "parque nacional", "red natura", "zepa", "zec",
    "gestión de residuos", "tratamiento de residuos",
    "autorización de vertido", "vertido de aguas",
    "plan hidrológico", "demarcación hidrográfica",
]
mask_b1 = df["description"].str.lower().str.contains("|".join(keywords_dominio), na=False)
df_b1 = df[mask_b1].copy()
print(f"Universo B1 estimado: {len(df_b1):,} registros ({len(df_b1)/len(df)*100:.2f}% del corpus)")
print(f"\nDistribución N1 en universo B1:")
print(df_b1["act_type_n1"].value_counts().head(10).to_string())


---

##  3. Ground truth — Muestreo estratificado

El ground truth se construye en dos pasos:
1. **Muestreo**: seleccionar registros representativos por categoría N2 (esta sección)
2. **Anotación**: etiquetar manualmente el CSV resultante (tarea externa)

### Cuotas de muestreo

| Categoría | Cuota | Señal principal |
|-----------|-------|----------------|
| AGU_GEN | 25 | confederación hidrográfica, aprovechamiento de aguas |
| AGU_RIE | 25 | regadío, comunidad de regantes |
| AGU_SND | 15 | sondeo para captación |
| AGU_ABS | 15 | abastecimiento de agua |
| VIA_PEC | 20 | vía pecuaria, cañada real |
| RES | 15 | gestión de residuos |
| MON | 10 | monte de utilidad pública |
| VER | 10 | autorización de vertido |
| ESP_NAT | 5 | parque natural, Red Natura, ZEPA |
| AGU_IND | 5 | aprovechamiento hidroeléctrico |
| PHD | 5 | plan hidrológico (si hay pool suficiente) |
| **Negativos** | **20** | sin ninguna de las keywords anteriores |


In [ ]:
# Pre-flight check: verificar pool disponible por label antes de samplear.
# Si alguna etiqueta tiene menos registros que la cuota, se reduce automaticamente.
import random
random.seed(SEED)

keywords_B1 = {
    "AGU_GEN": ["confederación hidrográfica", "comisaría de aguas", "aprovechamiento de aguas"],
    "AGU_RIE": ["regadío", "comunidad de regantes", "riego"],
    "AGU_SND": ["sondeo para captación", "captación de aguas subterráneas", "pozo de captación"],
    "AGU_ABS": ["abastecimiento de agua", "abastecimiento municipal", "agua potable"],
    "AGU_IND": ["aprovechamiento hidroeléctrico", "central hidroeléctrica", "refrigeración"],
    "VIA_PEC": ["vía pecuaria", "cañada real", "cordel", "vereda", "colada"],
    "MON":     ["monte de utilidad pública", "dominio público forestal", "ocupación de monte"],
    # ESP_NAT: keywords corregidas - "espacio natural protegido" y "reserva natural"
    # no devuelven resultados en el corpus Q1 2025
    "ESP_NAT": ["parque natural", "parque nacional", "red natura", "zepa", "zec", "zona de especial"],
    "RES":     ["gestión de residuos", "tratamiento de residuos", "planta de residuos"],
    "VER":     ["autorización de vertido", "vertido de aguas residuales", "dominio público hidráulico"],
    "PHD":     ["plan hidrológico", "demarcación hidrográfica"],
}

cuotas = {
    "AGU_GEN": 25, "AGU_RIE": 25, "AGU_SND": 15, "AGU_ABS": 15,
    "AGU_IND": 5,  "VIA_PEC": 20, "MON": 10,      "ESP_NAT": 5,
    "RES": 15,     "VER": 10,     "PHD": 5,
}
N_NEGATIVOS = 20

print(f"{'Label':<10} {'Pool':>8} {'Cuota':>7} {'Disponible':>11}")
print("-" * 42)
for label, kws in keywords_B1.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    pool_size = mask.sum()
    cuota = cuotas.get(label, 0)
    disponible = min(cuota, pool_size)
    estado = "OK" if pool_size >= cuota else f"REDUCIDA a {disponible}"
    print(f"{label:<10} {pool_size:>8,} {cuota:>7} {estado:>11}")


In [ ]:
# Muestreo estratificado por categoria.
# min(cuota, len(pool)) evita errores cuando el pool es menor que la cuota.
sampled_ids = set()
frames = []

for label, kws in keywords_B1.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas.get(label, 0), len(pool))
    if n == 0:
        print(f"  {label:<10} SKIP (pool vacio)")
        continue
    sample = pool.sample(n, random_state=SEED).copy()
    sample["grupo_muestreo"] = label
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {label:<10} pool={len(pool):>5,}  sampled={n}")

# Negativos: registros sin ninguna keyword del dominio B1
all_kws = [kw for kws in keywords_B1.values() for kw in kws]
mask_neg = ~df["description"].str.lower().str.contains("|".join(all_kws), na=False)
mask_neg = mask_neg & ~df.index.isin(sampled_ids)
negativos = df[mask_neg].sample(N_NEGATIVOS, random_state=SEED).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_muestreo = pd.concat(frames, ignore_index=True)
df_muestreo["id"] = range(len(df_muestreo))
print(f"\nTotal muestreado: {len(df_muestreo)} registros")


In [ ]:
# Validación del muestreo: 3 ejemplos por label para confirmar que pertenecen al dominio.
# Imprime el grupo, boletín y descripción truncada para revisión rápida.
print("Validación del muestreo - 3 ejemplos por grupo:\n")
for grupo in df_muestreo["grupo_muestreo"].unique():
    muestra = df_muestreo[df_muestreo["grupo_muestreo"] == grupo].head(3)
    print(f"--- {grupo} ---")
    for _, row in muestra.iterrows():
        desc = str(row["description"])[:130].replace("\n", " ")
        bul = str(row.get("bulletin", "?")).upper()
        print(f"  [{bul}] {desc}")
    print()


In [ ]:
# Guardar el muestreo sin anotar - la anotación manual es el siguiente paso
PATH_MUESTREO = "../data/ground_truth/ground_truth_B1_muestreo.csv"
Path(PATH_MUESTREO).parent.mkdir(parents=True, exist_ok=True)

# Columnas de anotación preparadas vacías para rellenar manualmente
df_muestreo["is_relevant_gt"]   = ""
df_muestreo["categories_gt"]    = ""
df_muestreo["subcategories_gt"] = ""
df_muestreo["notas_anotador"]   = ""

df_muestreo[["id","bulletin","description","grupo_muestreo",
             "is_relevant_gt","categories_gt","subcategories_gt","notas_anotador"]].to_csv(
    PATH_MUESTREO, index=False
)
print(f"Guardado: {PATH_MUESTREO}  ({len(df_muestreo)} registros)")
print("Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt")


### Dataset anotado

Una vez completada la anotación manual de `ground_truth_B1_muestreo.csv`, cargar con:

```python
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B1_anotado.csv")
```

La anotación debe cubrir las columnas `is_relevant_gt`, `categories_gt` (valores separados por coma, ej. `AGU_RIE,ESP_NAT`) y `subcategories_gt` (opcional).


---

##  4. Agente base

La lógica de clasificación vive en `src/clasificador/`. El notebook solo importa y orquesta.
`build_agent` acepta `output_type` y `prompt_registry` para soportar bloques distintos de B0.


In [ ]:
# Importar la capa de agente compartida y los prompts/schema propios de B1
from clasificador.agent import build_agent, run_experiment, clasificar_async
from clasificador.prompts_B1 import SYSTEM_PROMPT_B1_V1, PROMPT_REGISTRY as PROMPT_REGISTRY_B1
from clasificador.schema_B1 import ClassifierOutput as ClassifierOutputB1
from tqdm.asyncio import tqdm_asyncio


In [ ]:
# Construir el agente B1 con el prompt baseline (v1)
# Se pasan output_type y prompt_registry propios de B1 - agent.py permanece genérico
agent_b1_v1 = build_agent(
    model, "v1",
    output_type=ClassifierOutputB1,
    prompt_registry=PROMPT_REGISTRY_B1,
)


In [ ]:
# Validación cualitativa - 5 casos representativos del dominio B1
casos_b1 = [
    ("Resolución de la Confederación Hidrográfica del Duero por la que se otorga "
     "concesión de aguas superficiales para riego de 120 ha en Valladolid. "
     "Comunidad de Regantes del Canal de Macías Picavea.", "bocyl"),
    ("Anuncio de información pública sobre solicitud de autorización de vía pecuaria "
     "cañada real para construcción de línea eléctrica subterránea en Cáceres.", "doe"),
    ("Resolución de la Dirección General de Medio Natural por la que se aprueba "
     "el plan de gestión de la Zona de Especial Conservación (ZEC) ES4110108.", "bocyl"),
    ("Resolución de la Consejería de Hacienda por la que se convocan plazas de "
     "auxiliar administrativo en la Junta de Castilla y León.", "bocyl"),
    ("Anuncio de la Confederación Hidrográfica del Tajo sobre autorización de "
     "vertido de aguas residuales tratadas al río Jarama en Guadalajara.", "bocm"),
]

print("Validación cualitativa - 5 casos B1\n")
for i, (desc, bul) in enumerate(casos_b1, 1):
    msg = f"Boletín: {bul.upper()} (ámbito: {get_ambito(bul)})\n\nDescripción: {desc}"
    result = await agent_b1_v1.run(msg)
    r = result.output
    cats = [c.value for c in r.categories]
    subs = [s.value for s in r.subcategories]
    print(f"Caso {i} | relevant={r.is_relevant} | cats={cats} | subs={subs}")
    print(f"  Reasoning: {r.reasoning[:120]}")
    print()


---

##  5. Experimento 1 - Baseline zero-shot

**Problema**: no existe un clasificador para el dominio hídrico/natural. Necesitamos una línea base que mida el rendimiento zero-shot antes de cualquier optimización.

**Objetivo**: establecer el Macro-F1 de referencia con el prompt mínimo operativo (V1) y detectar los patrones de error sistemáticos que guiarán la siguiente iteración del prompt.

**Enfoque**: Qwen 3.5 9B · `SYSTEM_PROMPT_B1_V1` · zero-shot · sin contexto N1 · 180 registros.

**Resultados**:

| Métrica | Valor |
|---------|-------|
| is_rel F1 | - |
| Micro F1 | - |
| Macro F1 | - |
| Hamming Loss | - |
| Jaccard | - |
| Subset Acc | - |


In [ ]:
# TODO: ejecutar tras completar la anotación manual de ground_truth_B1_muestreo.csv
# y guardarla como ground_truth_B1_anotado.csv

# df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B1_anotado.csv")
# df_anotado_run = df_anotado[df_anotado["is_relevant_gt"] != ""]  # solo filas anotadas

# df_exp_b1_1 = await run_experiment(
#     agent_b1_v1, df_anotado_run,
#     use_n1_context=False, concurrency=1,
#     output_path="../results/b1_exp1_baseline_qwen9b.csv",
#     desc="B1 Exp1 - Baseline",
# )


---

##  6. Análisis de errores - Baseline

Evaluamos las predicciones del Exp 1 contra el ground truth para identificar patrones de error
que guíen la mejora del prompt en §7.


In [ ]:
# Convierte cualquier representación de etiquetas a un set de strings.
# Misma función que B0 - compatible con CSV manual ("AGU_RIE,ESP_NAT")
# y con JSON de predicciones ('["AGU_RIE","ESP_NAT"]').
def parse_labels(value) -> set:
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    v = str(value).strip()
    if v.startswith("["):
        try:
            return set(json.loads(v))
        except Exception:
            pass
    return set(x.strip() for x in v.split(",") if x.strip())


In [ ]:
def compute_metrics_B1(df_eval, label="", verbose=True):
    """
    Calcula métricas multilabel para el clasificador B1.
    Identica a compute_metrics de B0 pero con las etiquetas N2 de B1.
    Columnas esperadas: is_relevant_gt, categories_gt, is_relevant_pred, categories_pred.
    """
    # Etiquetas N2 de B1 - derivadas del enum para evitar hardcodear
    from clasificador.schema_B1 import CategoryType
    N2 = [e.value for e in CategoryType]

    y_true = df_eval["is_relevant_gt"].astype(bool)
    y_pred = df_eval["is_relevant_pred"].fillna(False).astype(bool)

    # -- Subproblema binario: is_relevant --
    tp = ((y_true) & (y_pred)).sum()
    fp = ((~y_true) & (y_pred)).sum()
    fn = ((y_true) & (~y_pred)).sum()
    tn = ((~y_true) & (~y_pred)).sum()
    prec_rel = tp / (tp + fp) if tp + fp > 0 else 0.0
    rec_rel  = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1_rel   = 2 * prec_rel * rec_rel / (prec_rel + rec_rel) if prec_rel + rec_rel > 0 else 0.0
    acc_rel  = accuracy_score(y_true, y_pred)

    if verbose:
        if label:
            print(f"-- {label} --")
        print(f"\nis_relevant  Acc={acc_rel:.3f}  P={prec_rel:.3f}  R={rec_rel:.3f}  F1={f1_rel:.3f}")
        print(f"             TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")

    # -- Multilabel N2: construir matrices binarias --
    Y_true = np.array([
        [1 if lbl in parse_labels(r["categories_gt"])   else 0 for lbl in N2]
        for _, r in df_eval.iterrows()
    ])
    Y_pred = np.array([
        [1 if lbl in parse_labels(r["categories_pred"]) else 0 for lbl in N2]
        for _, r in df_eval.iterrows()
    ])

    micro_f1   = f1_score(Y_true, Y_pred, average="micro",    zero_division=0)
    macro_f1   = f1_score(Y_true, Y_pred, average="macro",    zero_division=0)
    hl         = hamming_loss(Y_true, Y_pred)
    jaccard    = jaccard_score(Y_true, Y_pred, average="samples", zero_division=0)
    subset_acc = accuracy_score(Y_true, Y_pred)

    if verbose:
        print(f"N2 multilabel:")
        print(f"  Micro F1:      {micro_f1:.3f}")
        print(f"  Macro F1:      {macro_f1:.3f}")
        print(f"  Hamming Loss:  {hl:.4f}")
        print(f"  Jaccard:       {jaccard:.3f}")
        print(f"  Subset Acc:    {subset_acc:.3f}\n")

        print(f"{'Label':<10} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5}")
        print("-" * 35)
        for i, lbl in enumerate(N2):
            p2  = precision_score(Y_true[:, i], Y_pred[:, i], zero_division=0)
            r2  = recall_score   (Y_true[:, i], Y_pred[:, i], zero_division=0)
            f12 = f1_score       (Y_true[:, i], Y_pred[:, i], zero_division=0)
            print(f"{lbl:<10} {p2:>6.3f} {r2:>6.3f} {f12:>6.3f} {Y_true[:, i].sum():>5}")
        print("-" * 35)
        print(f"{'Macro':<10} {'':>13} {macro_f1:>6.3f}")

    card      = Y_true.sum(axis=1)
    exact_arr = (Y_true == Y_pred).all(axis=1)
    exact_ser = pd.Series(exact_arr, index=df_eval.index)

    if verbose:
        print("\nSubset Acc por cardinalidad:")
        for c, lbl_c in [(0, "card=0 (no relevante)"), (1, "card=1"), (2, "card=2"), (3, "card>=3")]:
            mask = (card >= c) if c == 3 else (card == c)
            if mask.sum() > 0:
                print(f"  {lbl_c:<22}: {exact_arr[mask].mean():.3f}  ({exact_arr[mask].sum()}/{mask.sum()})")
        conf = df_eval["confidence"].dropna()
        print(f"\nConfianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return {
        "is_rel_accuracy":  float(acc_rel),
        "is_rel_precision": float(prec_rel),
        "is_rel_recall":    float(rec_rel),
        "is_rel_f1":        float(f1_rel),
        "micro_f1":         float(micro_f1),
        "macro_f1":         float(macro_f1),
        "hamming_loss":     float(hl),
        "jaccard_samples":  float(jaccard),
        "subset_accuracy":  float(subset_acc),
        "exact":            exact_ser,
        "rel":              y_true,
    }


In [ ]:
# Muestra los registros relevantes donde la predicción N2 no coincide con el GT.
def print_errors(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    print(f"-- Errores N2 en relevantes{' · ' + label if label else ''} --")
    print(f"Total: {(~exact & rel).sum()}\n")
    for _, row in errores.iterrows():
        gt   = parse_labels(row["categories_gt"])
        pred = parse_labels(row["categories_pred"])
        print(f"ID {row['id']} | GT={sorted(gt)} | PRED={sorted(pred)}")
        print(f"  Falta: {sorted(gt - pred)} | Sobra: {sorted(pred - gt)}")
        print(f"  {str(row['description'])[:100]}...")
        print(f"  Razonamiento: {str(row.get('reasoning', ''))[:150]}")
        print()


In [ ]:
# TODO: ejecutar tras completar la anotación y el Experimento 1

# df_b1_exp1 = pd.read_csv("../results/b1_exp1_baseline_qwen9b.csv")
# df_eval_b1_1 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
#     df_b1_exp1[["description","is_relevant_pred","act_type_pred","categories_pred",
#                 "subcategories_pred","confidence","reasoning"]], on="description", how="left"
# )
# m_b1_1 = compute_metrics_B1(df_eval_b1_1, "Experimento B1-1 - Baseline")


In [ ]:
# TODO: descomentar tras tener resultados
# print_errors(df_eval_b1_1, m_b1_1["exact"], m_b1_1["rel"], label="B1 Experimento 1 - Baseline")


---

##  7. Experimento 2 - Prompt v2

**Problema**: (rellenar tras el análisis de errores del §6)

**Objetivo**: ¿mejoran las reglas explícitas el rendimiento en el dominio hídrico/natural?

**Enfoque**: pendiente de definir tras ver los errores del baseline.

**Resultados**: pendiente.

> ⚙️ **TODO**: definir `SYSTEM_PROMPT_B1_V2` en `prompts_B1.py` tras analizar los errores del Experimento 1.
> Los errores sistemáticos del baseline determinarán qué reglas añadir.


In [ ]:
# TODO: implementar SYSTEM_PROMPT_B1_V2 en prompts_B1.py tras analizar errores del Exp 1
# agent_b1_v2 = build_agent(model, "v2", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)
# df_b1_exp2 = await run_experiment(
#     agent_b1_v2, df_anotado,
#     use_n1_context=False, concurrency=1,
#     output_path="../results/b1_exp2_promptv2_qwen9b.csv",
#     desc="B1 Exp2 - Prompt v2",
# )


---

##  8. Experimento 3 - Few-shot

**Problema**: (rellenar tras el análisis de errores del §6 y §7)

**Objetivo**: ¿eliminan ejemplos quirúrgicos los errores sistemáticos sin degradar el resto?

**Enfoque**: pendiente — los ejemplos few-shot se seleccionarán de los errores del Exp 1 y Exp 2.

**Resultados**: pendiente.

> ⚙️ **TODO**: seleccionar 3-4 ejemplos de los fallos más frecuentes del Exp 1/Exp 2 y añadirlos
> como `SYSTEM_PROMPT_B1_V3` en `prompts_B1.py`.


In [ ]:
# TODO: implementar SYSTEM_PROMPT_B1_V3 con few-shot quirúrgico
# agent_b1_v3 = build_agent(model, "v3", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)
# df_b1_exp3 = await run_experiment(...)


---

##  9. Comparativa de modelos

**Problema**: ¿dependen los resultados B1 del modelo o el prompt es transferible a modelos más pequeños?

**Objetivo**: comparar Gemma 4 4B con el mejor prompt B1 para medir transferibilidad.

**Enfoque**: pendiente — ejecutar tras tener el mejor prompt de los Exp 1-3.

**Resultados**: pendiente.

> ⚙️ **TODO**: cargar `gemma-4-e4b-it` en LM Studio y ejecutar con el mejor prompt B1.


In [ ]:
# TODO: cargar gemma-4-e4b-it en LM Studio antes de ejecutar
# from pydantic_ai.models.openai import OpenAIChatModel
# from pydantic_ai.providers.openai import OpenAIProvider as OAIProvider
# model_gemma = OpenAIChatModel("gemma-4-e4b-it", provider=OAIProvider(base_url="...", api_key="lm-studio"))
# agent_b1_gemma = build_agent(model_gemma, "v2", output_type=ClassifierOutputB1, prompt_registry=PROMPT_REGISTRY_B1)
# df_b1_exp_gemma = await run_experiment(...)


---

##  10. Tabla resumen - Comparativa de experimentos B1

In [ ]:
# TODO: rellenar experimentos_cfg tras ejecutar los experimentos
# experimentos_cfg_B1 = [
#     ("B1 Exp1 - Baseline",    "Qwen 3.5 9B", "Zero-shot", "../results/b1_exp1_baseline_qwen9b.csv"),
#     ("B1 Exp2 - Prompt v2",   "Qwen 3.5 9B", "Zero-shot", "../results/b1_exp2_promptv2_qwen9b.csv"),
#     ("B1 Exp3 - Few-shot",    "Qwen 3.5 9B", "Few-shot",  "../results/b1_exp3_fewshot_qwen9b.csv"),
#     ("B1 Exp4 - Gemma 4B",    "Gemma 4 4B",  "Zero-shot", "../results/b1_exp4_gemma4b.csv"),
# ]
#
# rows_b1 = []
# metrics_list_b1 = []
# for nombre, modelo, config, path in experimentos_cfg_B1:
#     df_r = pd.read_csv(path)
#     df_e = df_anotado[["id","is_relevant_gt","categories_gt","description"]].merge(
#         df_r[["description","is_relevant_pred","categories_pred","confidence","reasoning"]],
#         on="description", how="left"
#     )
#     m = compute_metrics_B1(df_e, verbose=False)
#     metrics_list_b1.append(m)
#     rows_b1.append({
#         "Experimento": nombre, "Modelo": modelo, "Config": config,
#         "is_rel_f1": m["is_rel_f1"], "micro_f1": m["micro_f1"],
#         "macro_f1": m["macro_f1"], "hamming_loss": m["hamming_loss"],
#         "jaccard_samples": m["jaccard_samples"], "subset_accuracy": m["subset_accuracy"],
#         "errores_fmt": int(df_e["reasoning"].astype(str).str.startswith("ERROR").sum()),
#     })
# df_summary_b1 = pd.DataFrame(rows_b1)
# ...
